### Finstack Core Basics (Python)

This notebook demonstrates the core features of the Finstack Python bindings:

- Currency lookup and properties
- Money construction, formatting, scaling, and tuple round-trips
- Calendars, business day adjustment, and day count calculation
- ScheduleBuilder with business-day adjustments and end-of-month handling
- Calendar and fiscal period construction
- Date utilities
- Market data primitives and `MarketContext`

#### Prerequisites
- Build and install the Python extension so the `finstack` module is available to Python. For example:

```bash
uv run maturin develop --release
```

Then launch this notebook (or run with your preferred environment).


In [1]:
from __future__ import annotations

from datetime import date
import logging

from finstack.core.config import FinstackConfig
from finstack.core.currency import Currency
from finstack.core.dates import (
    BusinessDayConvention,
    DayCount,
    DayCountContext,
    FiscalConfig,
    Frequency,
    ScheduleBuilder,
    add_months,
    adjust,
    available_calendar_codes,
    build_fiscal_periods,
    build_periods,
    date_to_days_since_epoch,
    days_in_month,
    get_calendar,
    last_day_of_month,
    next_imm,
)
from finstack.core.market_data import (
    BaseCorrelationCurve,
    DiscountCurve,
    DividendScheduleBuilder,
    FxConfig,
    FxConversionPolicy,
    FxMatrix,
    HazardCurve,
    MarketContext,
    MarketScalar,
    ScalarTimeSeries,
    SeriesInterpolation,
    VolSurface,
)
from finstack.core.money import Money

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)



In [2]:
# Currency basics
usd = Currency("USD")
print("Code:", usd.code, "Numeric:", usd.numeric, "Decimals:", usd.decimals)

gbp = Currency.from_numeric(826)
print("Lookup by numeric code:", gbp)

print("Total built-in currencies:", len(Currency.all()))


Code: USD Numeric: 840 Decimals: 2
Lookup by numeric code: GBP
Total built-in currencies: 157


In [3]:
# Money basics
usd = Currency("USD")
amount = Money(1234.567, usd)
print("Raw:", amount.format())

config = FinstackConfig()
config.set_output_scale(usd, 4)
print("Formatted (custom scale):", amount.format_with_config(config))

subtotal = amount + Money(10.0, usd)
print("Addition:", subtotal.format())

# Tuple round-trip
as_tuple = subtotal.to_tuple()
print("Tuple:", as_tuple)
reconstructed = Money.from_tuple(as_tuple)
print("Round-trip equal:", reconstructed == subtotal)


Raw: USD 1234.57
Formatted (custom scale): USD 1234.5700
Addition: USD 1244.57
Tuple: (1244.57, Currency('USD'))
Round-trip equal: True


In [4]:
# Calendars and day count
codes = available_calendar_codes()
print("Available calendar sample:", codes[:5])

calendar = get_calendar("usny")
print("Calendar:", calendar.name, "(ignores weekends:", calendar.ignore_weekends, ")")

start = date(2025, 1, 4)  # Saturday
adjusted = adjust(start, BusinessDayConvention.FOLLOWING, calendar)
print("Adjusted business day:", adjusted)

dc_ctx = DayCountContext(calendar=calendar, frequency=Frequency.SEMI_ANNUAL)
yf = DayCount.ACT_ACT_ISMA.year_fraction(date(2025, 1, 4), adjusted, dc_ctx)
print("Year fraction (Act/Act ISMA):", round(yf, 6))

next_roll = next_imm(start)
print("Next IMM date:", next_roll)


Available calendar sample: ['asx', 'auce', 'brbd', 'cato', 'chzh']
Calendar: United States (New York Federal) Holidays (ignores weekends: False )
Adjusted business day: 2025-01-06
Year fraction (Act/Act ISMA): 0.01105
Next IMM date: 2025-03-19


In [5]:
# ScheduleBuilder example
start = date(2025, 1, 15)
end = date(2025, 7, 15)
calendar = get_calendar("usny")

schedule = (
    ScheduleBuilder.new(start, end)
    .frequency(Frequency.MONTHLY)
    .adjust_with(BusinessDayConvention.MODIFIED_FOLLOWING, calendar)
    .end_of_month(True)
    .build()
)

print("Generated schedule (business-day adjusted, EOM):")
for anchor in schedule.dates:
    print(" ", anchor)


Generated schedule (business-day adjusted, EOM):
  2025-01-31
  2025-02-28
  2025-03-31
  2025-04-30
  2025-05-30
  2025-06-30
  2025-07-31


In [6]:
# Periods and fiscal periods
plan = build_periods("2024Q1..Q3", actuals_until="2024Q2")
print("Calendar periods:")
for period in plan.periods:
    print(" ", period.id.code, ":", period.start, "->", period.end, "(actual=", period.is_actual, ")")

fiscal_plan = build_fiscal_periods("2025Q1..Q4", FiscalConfig.US_FEDERAL, None)
print("US Federal fiscal periods:")
for period in fiscal_plan.periods:
    print(" ", period.id.code, ":", period.start, "->", period.end)


Calendar periods:
  2024Q1 : 2024-01-01 -> 2024-04-01 (actual= True )
  2024Q2 : 2024-04-01 -> 2024-07-01 (actual= True )
  2024Q3 : 2024-07-01 -> 2024-10-01 (actual= False )
US Federal fiscal periods:
  2025Q1 : 2024-10-01 -> 2025-01-01
  2025Q2 : 2025-01-01 -> 2025-04-01
  2025Q3 : 2025-04-01 -> 2025-07-01
  2025Q4 : 2025-07-01 -> 2025-10-01


In [7]:
# Date utilities
base = date(2025, 1, 30)
print("Add months:", add_months(base, 1))
print("Last day of month:", last_day_of_month(base))
print("Days in Feb 2024:", days_in_month(2024, 2))
print("Days since epoch:", date_to_days_since_epoch(base))


Add months: 2025-02-28
Last day of month: 2025-01-31
Days in Feb 2024: 29
Days since epoch: 20118


In [8]:
# Market data primitives and MarketContext
base = date(2024, 1, 2)
usd = Currency("USD")

discount = DiscountCurve(
    "USD-OIS",
    base,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85)],
    day_count=DayCount.ACT_365F,
    interp="monotone_convex",
)

hazard = HazardCurve(
    "CDX-IG",
    base,
    [(0.0, 0.01), (5.0, 0.015), (10.0, 0.02)],
    recovery_rate=0.4,
    currency=usd,
    day_count=DayCount.ACT_365F,
)

base_corr = BaseCorrelationCurve(
    "CDX-IG",
    [(3.0, 0.25), (7.0, 0.45), (10.0, 0.6)],
)

surface = VolSurface(
    "EQ-FLAT",
    expiries=[1.0, 2.0],
    strikes=[90.0, 100.0, 110.0],
    grid=[[0.2, 0.21, 0.22], [0.19, 0.2, 0.21]],
)

fx = FxMatrix(config=FxConfig(enable_triangulation=True))
fx.set_quote(Currency("EUR"), usd, 1.1)
fx_rate = fx.rate(Currency("EUR"), usd, base, FxConversionPolicy.CASHFLOW_DATE)

scalar = MarketScalar.price(Money(188.25, usd))
series = ScalarTimeSeries(
    "US-CPI",
    [(date(2023, 12, 31), 300.0), (date(2024, 1, 31), 301.5)],
    interpolation=SeriesInterpolation.LINEAR,
)

dividends_builder = DividendScheduleBuilder("AAPL-DIVS")
dividends_builder.underlying("AAPL")
dividends_builder.cash(date(2024, 2, 15), Money(0.24, usd))
dividends_builder.cash(date(2024, 5, 15), Money(0.25, usd))
dividends = dividends_builder.build()

context = MarketContext()
context.insert_discount(discount)
context.insert_hazard(hazard)
context.insert_base_correlation(base_corr)
context.insert_surface(surface)
context.insert_price("AAPL", scalar)
context.insert_series(series)
context.insert_dividends(dividends)
context.insert_fx(fx)

stats = context.stats()
print("Context stats:", stats)
discount_curve = context.discount("USD-OIS")
print("Discount df(5y):", round(discount_curve.df(5.0), 4))
print("EUR/USD fx rate (triangulated=", fx_rate.triangulated, "):", round(fx_rate.rate, 4))


Context stats: {'total_curves': 2, 'surface_count': 1, 'price_count': 1, 'series_count': 1, 'inflation_index_count': 0, 'credit_index_count': 0, 'dividend_schedule_count': 1, 'collateral_mapping_count': 0, 'has_fx': True, 'curve_counts': {'BaseCorrelation': 1, 'Discount': 1}}
Discount df(5y): 0.85
EUR/USD fx rate (triangulated= False ): 1.1


In [9]:
context.discount("USD-OIS").df(4.0)

0.8787917794452795